# 종합 실습 - LLM부터 LangGraph까지

00~04에서 배운 내용을 복습하는 실습입니다.
`___` 부분을 채워 넣으세요.

## 0. 환경 설정

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-5.4")
print("\u2713 모델 준비 완료")

✓ 모델 준비 완료


---
## 1단계: 메시지와 invoke (01 복습)

SystemMessage와 HumanMessage를 사용해서 모델을 호출하세요.

In [ ]:
from langchain.messages import SystemMessage, HumanMessage

# Q1. 시스템 메시지: "당신은 한국 역사 전문가입니다."
#     사용자 질문: "세종대왕의 업적을 3가지 알려주세요."
messages = [
    ___(content="당신은 한국 역사 전문가입니다."),
    ___(content="세종대왕의 업적을 3가지 알려주세요."),
]

# Q2. invoke로 모델 호출
response = model.___(messages)
print(response.____[:200])

---
## 2단계: 스트리밍 (01 복습)

stream()으로 실시간 출력하세요.

In [ ]:
# Q3. stream으로 토큰 단위 출력 (for문 사용)
print("스트리밍: ", end="")
for chunk in model.___("대한민국의 수도는 어디인가요?"):
    print(chunk.content, end="", flush=True)
print()

---
## 3단계: 도구 만들기 (02 복습)

@tool 데코레이터로 도구를 만드세요.

In [ ]:
from langchain.tools import tool

# Q4. @tool 데코레이터를 붙여 도구로 만드세요
___
def subtract(a: int, b: int) -> int:
    """두 수를 뺍니다."""
    return a - b

___
def divide(a: int, b: int) -> float:
    """두 수를 나눕니다."""
    return a / b

print(f"도구 이름: {subtract.name}, {divide.name}")

---
## 4단계: 에이전트 생성 (02 복습)

create_agent()로 도구를 가진 에이전트를 만드세요.

In [ ]:
from langchain.agents import create_agent

# Q5. 모델과 도구를 결합하여 에이전트 생성
agent = ___(
    model=model,
    tools=[subtract, divide],
    system_prompt="_____________",
)

# Q6. invoke로 에이전트 실행
result = agent.___({
    "messages": [{"role": "user", "content": "100에서 37을 빼주세요."}]
})
print("결과:", result["messages"][-1].____)

---
## 5단계: 메모리 에이전트 (03 복습)

InMemorySaver와 thread_id로 대화를 기억하는 에이전트를 만드세요.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

# Q7. checkpointer에 InMemorySaver()를 전달
memory_agent = create_agent(
    model=model,
    tools=[subtract, divide],
    checkpointer=___,
)

# Q8. thread_id로 세션 구분하는 config 작성
config = {"configurable": {"___": "practice-1"}}

# 첫 번째 질문
r1 = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "200 나누기 8은?"}]},
    config=config,
)
print("1차:", r1["messages"][-1].content)

# 두 번째 질문 - 이전 결과를 기억해야 함
r2 = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "그 결과에서 10을 빼주세요."}]},
    config=config,
)
print("2차:", r2["messages"][-1].content)

---
## 6단계: LangGraph 워크플로 (04 복습)

StateGraph로 2개의 노드를 연결하는 그래프를 만드세요.

흐름: `START → reverse_text → count_chars → END`

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

# Q9. State 정의 (text: str, char_count: int)
class State(___): 
    text: str
    char_count: int

# 노드 함수들
def reverse_text(state: State) -> dict:
    return {"text": state["text"][::-1]}

def count_chars(state: State) -> dict:
    return {"char_count": len(state["text"])}

# Q10. 그래프 빌드: 노드 등록 → 엣지 연결 → 컴파일
builder = StateGraph(State)
builder.___("_____", _____)       # 노드 등록
builder.___("_____", _____)           # 노드 등록
builder.___(START, "_____")               # 엣지 연결
builder.___("_____", "_____")             # 엣지 연결
builder.___("_____", END)                   # 엣지 연결

graph = builder.___()                        # 컴파일

result = graph.invoke({"text": "Hello LangGraph"})
print(f"원본 → 뒤집기: {result['text']}")
print(f"글자 수: {result['char_count']}")

---
## 수고하셨습니다!

정답은 `practice_answer.ipynb`에서 확인하세요.